# 固定2×2载波与随机载波：full-only用户运行包
四个已见host、原P0独立分叉RANDOM/BLOCK2；只换同步carrier vectors，同key dither/pilot/mask及QIM/RM/pilot/强度不变。OFF复用旧实际readback只新E作null；P0/R/B新同runtime D/量化/MP4/E。净32用户payload与A/default不变。
固定VAE1、D12、MP4 encode/probe/decode各12、E181×16、FFT12供两钥24读/Counter9216；预计552 decoder/736 encoder chunks。4224 tag行全保留，读出202752个64D dots，构造另33792 dots；仅同family同chip错钥96 dots。不做E41、phase/path、加幅/缩放、阈值调参、方向筛选或重试。
本包仅完成CPU/合成/stub工程核验；没有真实模型/GPU/Colab/媒体结果。四源皆VIEWED，无独立确认、FPR认证或科学PASS；旧tau仅描述。
运行源码来自GitHub的dev/local-joint-state-payload-v1分支。源码发布不等于实验已执行；GPU仍由用户Run all启动。Notebook直接clone可用源码，不要求手动ZIP、B64、摘要/manifest/版本/Git-clean门禁。
一次attempt和每call STARTED先落盘，失败/中断消费；后续Run all只读已有结果，不下载源码、装依赖、续算或重试。缺输入/模型失败留槽，不用旧媒体代替新P0或自动decode OFF。全部raw/payload/final保存并回读后才打开writer bits及quality后评；tag失败不跳payload。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, subprocess
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-Block2-Carrier-V1/fixed-viewed-full-v1')
def execution_artifacts(output):
    out=Path(output);found=[]
    for name in ('attempt.json','ledger.json','result.json','evaluation.json','plan.json','fixed_rows.json','progress.json','post_reference_access.json','tag_post_rows.json','tag_post_chips.json','carrier_full_pairs.json','payload_post_rows.json','payload_group_pairs.json','payload_cell_pairs.json'):
        for suffix in ('','.tmp'):
            p=out/(name+suffix)
            if p.exists() or p.is_symlink():found.append(name+suffix)
    for folder in ('raw','blind','payload','native','construction','latents','media','quality','markers','receipts','books'):
        p=out/folder
        if p.exists():found.extend(str(x.relative_to(out)) for x in p.rglob('*') if x.is_file() or x.is_symlink())
    return sorted(found)
EXISTING_ARTIFACTS=execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    SOURCE_ROOT=Path('/content/SC-SSTW-b-block2-carrier-v1')
    if not SOURCE_ROOT.exists():
        subprocess.run(['git','clone','--depth','1','--single-branch','--branch','dev/local-joint-state-payload-v1','https://github.com/RICHAAARC/SC-SSTW.git',str(SOURCE_ROOT)],check=True)
    if not (SOURCE_ROOT/'experiments/wan_state_clock/b_block2_carrier_v1_run.py').is_file():
        raise FileNotFoundError('Available source lacks the block2 entry: '+str(SOURCE_ROOT))
    OUTPUT.mkdir(parents=True,exist_ok=True)
    print('Available editable source',SOURCE_ROOT)
else:
    print('Existing attempt/artifacts: saved JSON read-only, no setup/model/physical reentry.')


运行固定实验。导入失败只修具体依赖；失败/中断的科学调用不重试。GPU由用户Run all启动。

In [ ]:

import subprocess, sys, time
if not EXISTING_ARTIFACTS:
    LOG = OUTPUT/'setup.log'
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode
    def ensure_imports(probe, packages, label):
        import re
        mapping = {p.replace('-', '_'):p for p in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1).split('.')[0] in mapping:
                package = mapping[missing.group(1).split('.')[0]]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' actual import failure; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, action)
    import shutil
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        command(['apt-get','update'], 'refresh package metadata for missing FFmpeg')
        command(['apt-get','install','-y','ffmpeg'], 'install missing ffmpeg/ffprobe')
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            raise RuntimeError('ffmpeg/ffprobe unavailable after dependency installation; see '+str(LOG))
    ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub; from diffusers import AutoencoderKLWan',
        ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan VAE dependencies')
    with (OUTPUT/'runner.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'experiments.wan_state_clock.b_block2_carrier_v1_run', '--output', str(OUTPUT)],
            cwd=SOURCE_ROOT, stdout=log, stderr=subprocess.STDOUT)
    try:
        while process.poll() is None:
            time.sleep(30)
            try:
                progress = json.loads((OUTPUT/'progress.json').read_text())
                print('Units', progress.get('counts',{}).get('RETURNED',0), '/', progress['planned'], 'counts', progress.get('counts',{}), 'active', progress.get('active'), flush=True)
            except (OSError,ValueError,KeyError,TypeError) as error:
                print('Progress read unavailable; observing same worker:',type(error).__name__,str(error),flush=True)
    except KeyboardInterrupt:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill(); process.wait()
        print('Stopped; saved attempts consumed. Later Run all is read-only.')
        raise
    print('Runner return code',process.returncode,'; inspect saved results and runner.log. No retry.')
else:
    print('Existing attempt: source/setup/physical execution skipped.')


只读结果：标签、实际扰动、载荷退步与画质分别报告，不据此自动选方案。

In [ ]:

for name in ('result.json','evaluation.json'):
    p=OUTPUT/name
    if not p.is_file():print('Unavailable',name);continue
    try:v=json.loads(p.read_text())
    except (OSError,ValueError) as e:print('Saved read failure',name,str(e));continue
    if name=='result.json':print('Execution',v.get('status'),'calls',v.get('kinds_started'),'technical tag rows',v.get('technical_tag_rows'))
    else:
        print('Fixed denominators',v.get('fixed_denominators'))
        for k,x in v.get('tag_summary',{}).items():print('Tag',k,x)
        print('Payload',v.get('payload_total'),'truth issues',v.get('reference_issues'))
print('Construction receipts: actual FP32 energy/RMS/Linf, coefficients and projection errors in construction/.')
print('Quality24 paired preclip/RGB8/MP4 outputs and clipping in quality/; no adopted quality PASS threshold.')
print('All failures remain; full clean-span payload is conditional, not tag-guided. No local/unknownphase/FPR/independent confirmation claim. Output:',OUTPUT)
